In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [23]:
## https://stats.stackexchange.com/questions/205932/dropout-scaling-the-activation-versus-inverting-the-dropout

class LogisticDropout(nn.Module):
    def __init__(self,
                 input_size: int,
                 r: float = 3.9,
                 seed: int = None):
        super(LogisticDropout, self).__init__()

        # constants
        self.r = r
        self.seed = seed

        # intialize state (0, 1]
        if self.seed is not None:
            torch.manual_seed(self.seed)
        self.logistic_map = lambda x: r * x * (1 - x)
        self.state = torch.rand(input_size)

    def forward(self, x):
        if self.training:
            self.state = self.logistic_map(self.state)
            return x * self.state.to(x.device) / 0.5
        return x

In [24]:
# test LogisticDropout on a random 3 dimensional tensor
if __name__ == '__main__':
    size = (2, 3)
    x = torch.rand(size)
    logistic_dropout = LogisticDropout(size, seed=0)
    print(x)
    print(logistic_dropout(x))

tensor([[0.4901, 0.8964, 0.4556],
        [0.6323, 0.3489, 0.4017]])
tensor([[0.9556, 1.2450, 0.2866],
        [0.5652, 0.5794, 0.7270]])


In [76]:
for r in range(21, 45, 1):
    # check what the average of the logistic map is after a few iterations
    averages = []
    size = (1000, 1000)
    logistic_dropout = LogisticDropout(size, r=r/10, seed=0)
    for i in range(1000):
        logistic_dropout.state = logistic_dropout.logistic_map(logistic_dropout.state)
        averages.append(logistic_dropout.state.mean())
    print(r/10, sum(averages) / len(averages))

2.1 tensor(0.5234)
2.2 tensor(0.5451)
2.3 tensor(0.5649)
2.4 tensor(0.5830)
2.5 tensor(0.5997)
2.6 tensor(0.6151)
2.7 tensor(0.6293)
2.8 tensor(0.6426)
2.9 tensor(0.6549)
3.0 tensor(0.6661)
3.1 tensor(0.6612)
3.2 tensor(0.6562)
3.3 tensor(0.6515)
3.4 tensor(0.6470)
3.5 tensor(0.6463)
3.6 tensor(0.6464)
3.7 tensor(0.6679)
3.8 tensor(0.6418)
3.9 tensor(0.5920)
4.0 tensor(0.4748)
4.1 tensor(-inf)
4.2 tensor(-inf)
4.3 tensor(-inf)
4.4 tensor(-inf)


tensor(0.9749)
tensor(0.0953)
tensor(0.3361)
tensor(0.8703)
tensor(0.4403)
tensor(0.9611)
tensor(0.1459)
tensor(0.4859)
tensor(0.9742)
tensor(0.0979)
